In [24]:
! nvidia-smi

Sat Feb 22 21:33:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             26W /   70W |    9358MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 환경 설정

In [25]:
!pip3 install -q -U bitsandbytes
!pip3 install -q -U peft
!pip3 install -q -U trl==0.12.0
!pip3 install -q -U accelerate
!pip3 install -q -U datasets
!pip3 install -q -U transformers

## 시작하기 전 허깅페이스 토큰 설정

In [ ]:
from huggingface_hub import login

TOKEN = '' # @param { "type": "string"}
login(TOKEN)

모델 그냥 로드하면 무거우므로 양자화한 상태로 로드 ( BitsAndBytes )

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "carrotter/ko-gemma-2b-it-sft"
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})
tokenizer = AutoTokenizer.from_pretrained(model_id, add_eos_token=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/522 [00:00<?, ?B/s]

In [ ]:
## 실행 간편히 하기위한 레퍼함수
def get_answer( query: str, model, tokenizer, max_tokens=100 ) -> str:
  chat = [
    { "role": "user", "content": query },
  ]
  prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

  inputs = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")
  outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=max_tokens)
  return (tokenizer.decode(outputs[0]))

In [ ]:
result = get_answer(query = "삶이란 무엇일까요?", model=model, tokenizer=tokenizer, max_tokens=200)
print(result)

<bos><start_of_turn>user
삶이란 무엇일까요?<end_of_turn>
<start_of_turn>model
삶이란 개인의 생활과 경험, 그리고 그들이 살고 있는 사회에서 그들이 받는 의미를 포함합니다. 이러한 의미는 개인의 삶의 목적, 가치관, 그리고 그들이 경험한 성공과 실패에 따라 달라질 수 있습니다. 삶은 개인이 자신의 삶을 살아가며, 그들이 가진 가치와 목적을 추구하는 과정입니다.<end_of_turn>
<end_of_turn>model
삶은 개인의 생활과 경험, 그리고 그들이 살고 있는 사회에서 그들이 받는 의미를 포함합니다. 이러한 의미는 개인의 삶의 목적, 가치관, 그리고 그들이 경험한 성공과 실패에 따라 달라질 수 있습니다. 삶은 개인이 자신의 삶을 살아가며, 그들이 가진 가치와 목적을 추구하는 과정입니다.<end_of_turn>
이러한 개념


In [ ]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='./schopenhauer_qna.csv', split='train')
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['input', 'output'],
    num_rows: 52
})

In [ ]:
def generate_prompt(data_point):
  prompt_template = '''
  <start_of_turn>user
  {input}
  <end_of_turn>\n<start_of_turn>model
  {output}
  '''

  return prompt_template.format(
      input=data_point["input"],
      output=data_point["output"])


def generate_and_tokenize_prompt(data_point):
  full_prompt = generate_prompt(data_point)
  tokenized_full_prompt = tokenizer(full_prompt)
  return tokenized_full_prompt

generate_prompt({'input':'test','output':'tst2'})

'\n  <start_of_turn>user\n  test\n  <end_of_turn>\n<start_of_turn>model\n  tst2\n  '

In [ ]:
dataset = dataset.add_column('prompt', [generate_prompt(row) for row in dataset])
dataset

Dataset({
    features: ['input', 'output', 'prompt'],
    num_rows: 52
})

In [ ]:
dataset[0]

{'input': '삶은 왜 이렇게 힘든 걸까요?',
 'output': '삶은 고통과 결핍으로 이루어져 있다네. 만족은 일시적일 뿐, 새로운 결핍이 다시 나타나기 마련이지.',
 'prompt': '\n  <start_of_turn>user\n  삶은 왜 이렇게 힘든 걸까요?\n  <end_of_turn>\n<start_of_turn>model\n  삶은 고통과 결핍으로 이루어져 있다네. 만족은 일시적일 뿐, 새로운 결핍이 다시 나타나기 마련이지.\n  '}

In [ ]:
dataset = dataset.map(lambda samples : tokenizer(samples['prompt']), batched=True)
dataset

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output', 'prompt', 'input_ids', 'attention_mask'],
    num_rows: 52
})

# LoRA 적용

In [ ]:
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
print(model)

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (n

In [ ]:
# LoRA는 트랜스포머 특정 레이어에 붙어서 파인튜닝하는 테크닉인데 로라 모듈이 붙을 수 있는
# 레이어를 찾는다.

import bitsandbytes as bnb
def find_all_linear_names(model):
  cls = bnb.nn.Linear4bit # if args.bits == 4 else (bnb.nn.Linear8bitLt if args.bits == 8 else torch.nn.Linear)
  lora_module_names = set()
  for name, module in model.named_modules():
    if isinstance(module, cls):
      names = name.split('.')
      lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names: # needed for 16-bit
      lora_module_names.remove('lm_head')
  return list(lora_module_names)

In [ ]:
modules = find_all_linear_names(model)
print(modules)

['down_proj', 'v_proj', 'gate_proj', 'k_proj', 'o_proj', 'up_proj', 'q_proj']


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


In [ ]:
model.print_trainable_parameters()

trainable params: 78,446,592 || all params: 2,584,619,008 || trainable%: 3.0351


# 학습

- Trainer : 큰 데이터셋과 복잡한 학습 워크 플로우에 적합
- SFTTrainer : 작은 데이터셋으로 사전 학습된 모델을 미세 조정하는 데 더 적합

In [ ]:
import transformers
from trl import SFTTrainer

tokenizer.pad_token = tokenizer.eos_token
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field='prompt',
    peft_config=lora_config,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=3,
        max_steps=110,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir='outputs',
        optim='paged_adamw_8bit',
        save_strategy='epoch',
        report_to="none",
    ),
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label

In [ ]:
model.config.use_cache = False # silence the warnings. Please re-enable for inference!
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,3.820900
2,3.911500
3,3.366200
4,2.836700
5,2.431900
6,2.338300
7,2.214300
8,2.152500
9,1.984600
10,1.921500


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/

TrainOutput(global_step=110, training_loss=0.5474336030131036, metrics={'train_runtime': 465.1246, 'train_samples_per_second': 0.946, 'train_steps_per_second': 0.236, 'total_flos': 335677369147392.0, 'train_loss': 0.5474336030131036, 'epoch': 8.461538461538462})

In [ ]:
new_model = "my_schopenhauer_model"
trainer.model.save_pretrained(new_model)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map='auto',
)
merged_model = PeftModel.from_pretrained(base_model, new_model)
merged_model = merged_model.merge_and_unload()

# 모델 병합
merged_model.save_pretrained("merged_model",  safe_serialization=True)
tokenizer.save_pretrained("merged_model")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# 파인튜닝된 모델 테스트

In [ ]:
result = get_answer(
    query = "삶이란 무엇일까요?",
    model=merged_model,
    tokenizer=tokenizer,
    max_tokens=200
)


print(result)

<bos><start_of_turn>user
삶이란 무엇일까요?<end_of_turn>
<start_of_turn>model
삶은 고통과 욕망의 화신이라네. 고통은 욕망을 불러오는 유일한 존재라네.
<end_of_turn>
<start_of_turn>model
삶은 고통과 욕망의 화신이라네.
<end_of_turn>
<end_of_turn>model
고통은 욕망을 불러오는 유일한 존재라네.
<end_of_turn>
<end_of_turn>model
삶은 고통과 욕망의 화신이라네.
<end_of_turn>
<end_of_turn>model<end_of_turn>
고통은 욕망을 불러오는 유일한 존재라네.
<end_of_turn>
<end_of_turn>model<end_of_turn>
삶은 고통과 욕망의 화신이라네.
<end_of_turn>
<end_of_turn>model<end_of_turn>
고통은 욕망을 불러오는 유일한 존재라네.
<end_of_turn>
<end_of_turn>
<end_of_turn>
고통은 욕망을 불러오는 유일한 존재라네


# 개인 모델 레지스트리에 푸쉬

실행 전, huggingface에서 모델 레지스트리를 생성하고 올 것.

In [ ]:
merged_model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.push_to_hub(new_model, use_temp_dir=False)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/PocaChip/my_schopenhauer_model/commit/eba10b390e81f4b06228d5164f0489316e8bf0eb', commit_message='Upload tokenizer', commit_description='', oid='eba10b390e81f4b06228d5164f0489316e8bf0eb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/PocaChip/my_schopenhauer_model', endpoint='https://huggingface.co', repo_type='model', repo_id='PocaChip/my_schopenhauer_model'), pr_revision=None, pr_num=None)

In [ ]:
!pip list

Package                            Version
---------------------------------- -------------------
absl-py                            1.4.0
accelerate                         1.4.0
aiohappyeyeballs                   2.4.6
aiohttp                            3.11.12
aiosignal                          1.3.2
alabaster                          1.0.0
albucore                           0.0.23
albumentations                     2.0.4
ale-py                             0.10.2
altair                             5.5.0
annotated-types                    0.7.0
anyio                              3.7.1
argon2-cffi                        23.1.0
argon2-cffi-bindings               21.2.0
array_record                       0.6.0
arviz                              0.20.0
astropy                            7.0.1
astropy-iers-data                  0.2025.2.17.0.34.13
astunparse                         1.6.3
atpublic                           4.1.0
attrs                              25.1.0
audioread          